# Task 1: Data preparation

### 1A: Exploratory data analysis

In [ ]:
# import packages
import numpy as np
import pandas as pd
import seaborn as sb

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-pastel')

In [ ]:
df_mood = pd.read_csv('dataset_mood_smartphone.csv')
df_mood

In [ ]:
print(df_mood.id.unique())

time_measures = []
for id in df_mood.id.unique():
    id_df = df_mood[df_mood.id == id]
    time_measures.append(len(id_df.time.unique()))

print(sorted(time_measures))

In [ ]:
df_mood.variable.unique()

#### 1A.1 Ranges of values

In [ ]:
for variable in df_mood.variable.unique():
    var_df = df_mood[df_mood.variable == variable]
    print(f'{variable}, min: {sorted(var_df.value)[0]}, max: {sorted(var_df.value)[-1]}')

#### 1A.2 Missing values

In [ ]:
# check whether there are any missing values in not 'value' column
df_mood.info()
# only in the last column, there are missing values
# 376912 - 376710 = 202 missing values

In [ ]:
# missing values per variable
variables = df_mood.variable.unique()
missing = []
for variable in variables:
    var_df = df_mood[df_mood.variable == variable]
    missing.append(var_df["value"].isnull().sum())

# missing list adds up to 202:
print(missing)
print(sum(missing))

# plot distribution for this

fig, ax = plt.subplots()

ax.bar(variables, missing)
ax.set_ylabel("Amount of missing values")
ax.set_xlabel("Variable")
ax.tick_params("x", rotation = 45)

plt.setp(ax.get_xticklabels(), ha = "right")
plt.tight_layout()

    

In [ ]:
# missing values per participant
ids = df_mood.id.unique()
missing = []

missing_counts = {'Arousal': [], 'Valence': []}

for id in ids:
    #define df for this id
    id_df = df_mood[df_mood.id == id]

    # count missing values in arousal
    id_arouse_df = id_df[id_df.variable == 'circumplex.arousal']
    missing_counts['Arousal'].append(id_arouse_df["value"].isnull().sum())

    #count missing values in valence
    id_valence_df = id_df[id_df.variable == 'circumplex.valence']
    missing_counts['Valence'].append(id_valence_df['value'].isnull().sum())
   
# missing list adds up to 202:
print(sum(missing_counts['Arousal']) + sum(missing_counts['Valence']))

# plot distribution for this

fig, ax = plt.subplots()
bottom = np.zeros(len(ids))

for variable, counts in missing_counts.items():
    p = ax.bar(ids, counts, label = variable, bottom = bottom)
    bottom += counts
    # ax.bar_label(p, label_type = 'center')

# ax.bar(ids, missing)
ax.set_ylabel("Amount of missing values")
ax.set_xlabel("Participant")
ax.tick_params("x", rotation = 45)
ax.legend()

plt.setp(ax.get_xticklabels(), ha = "right")
plt.tight_layout()

#### 1A.3 Distribution of values

In [ ]:
# disribution of all values 

fig, ax = plt.subplots()
ax.hist(df_mood.value)
ax.set_yscale('log', base = 10)
ax.set_xlabel(f'Values')
ax.set_ylabel('count')

In [ ]:
# distribution of ids

fig, ax = plt.subplots()
ax.hist(df_mood["Unnamed: 0"])
ax.set_yscale('log', base = 10)
ax.set_xlabel(f'IDs')
ax.set_ylabel('Count')

In [ ]:
# distribution of values for each variable
for variable in df_mood.variable.unique():
    var_df = df_mood[df_mood.variable == variable]

    fig, ax = plt.subplots()
    ax.hist(var_df.value, bins = 20)
    ax.set_yscale('log', base = 10)
    ax.set_xlabel(f'{variable}')
    ax.set_ylabel('Count')

#### 1A.4 Relationships between attributes

In [ ]:
# remove unusual instances before we do correlation testing
builtin = [-82798.871, -44.689, -1.218]
entertain = [-0.011]

# print(df_mood.loc[df_mood['value'] == builtin[0]])
# print(df_mood.loc[df_mood['value'] == builtin[1]])
# print(df_mood.loc[df_mood['value'] == builtin[2]])
# print(df_mood.loc[df_mood['value'] == entertain])

df_mood_filt = df_mood[~df_mood["value"].isin(builtin)]
df_mood_filt = df_mood_filt[~df_mood_filt["value"].isin(entertain)]

df_mood_filt = df_mood_filt.dropna(axis = 0)
df_mood_filt

In [ ]:
df_var_val = df_mood_filt.iloc[:, [3,4]]

df_pivoted = df_var_val.pivot_table(index=df_var_val.groupby('variable').cumcount(), 
                             columns='variable', 
                             values='value')

df_pivoted.columns.name = None
df_pivoted = df_pivoted.reset_index(drop=True)

# I want to move the mood column to the end of this dataframe
cols = [c for c in df_pivoted.columns if c != 'mood'] + ['mood']
df_pivoted = df_pivoted[cols]

df_pivoted



In [ ]:

corr_mtx = df_pivoted.corr(numeric_only=True)
print(corr_mtx)

corr_plot = sb.heatmap(corr_mtx, cmap = "coolwarm")
# plt.xticks([df_pivoted.columns])
# plt.tick_params("x", rotation = 45)
plt.yticks([])
plt.xlabel('Features')
plt.ylabel('Features')
plt.title('Heat map of the attributes globally')
plt.show()

Call and sms are not correlating with any other variables cause they are just 1s. 

#### 1A.5 Relationships between mood and attribute

This can be seen above, mood does not correlate highly with any of the other variables. 

#### 1A.6 Split dataset into participant data????

#### 1A Summary

There are 276911 instances (without headers) in the raw data and 5 colomns
- 27 participants

Amount of measures in time differs for the participants. Ranging from 2270 times, to 21298 times

19 attributes in the 'variable' column

### 1B: Data cleaning

First remove the outliers

The outliers are:
- appCat.builtin: three values: -82798.871 and -44.689 and -1.218
- appCat.entertainment: -0.011

I think they all need to be removed? Or we could choose to make them zero, but I think in the case of -82798.871 that is weird so maybe it would be better to just remove all of these.

In [ ]:
df_mood = pd.read_csv('dataset_mood_smartphone.csv')

# remove unusual instances before we do correlation testing
to_remove = [-82798.871, -44.689, -1.218, -0.011]

# check if these values occur only once
for value in to_remove:
    count = df_mood["value"].value_counts().get(value, 0)
    print(f"{value} appears {count} time(s)")


df_mood_filt = df_mood[~df_mood["value"].isin(to_remove)]
df_mood_filt

#### 1B.1 Imputing missing data

It says: select two approaches to imput missing values that are logical for such time series and argue for one of them based on the insights you gain. 

Consider what to do with prolonged periods of missing data in a time series.


In [ ]:
# Because of this last remark, first plot the data for the two variables that are missing values

# find participants for which there is data missing
# missing values per participant
ids = df_mood_filt.id.unique()
ids_withmiss = []

missing_counts = {'Arousal': [], 'Valence': []}

for id in ids:
    #define df for this id
    id_df = df_mood[df_mood.id == id]
    if id_df['value'].isnull().sum() > 0:
        ids_withmiss.append(id)

print(ids_withmiss)

# plot that data (lines?)



# later impute the data and compare

